# Imports 

In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import time
from tqdm import tqdm

# IMDB Part

## Part 1 - Loading The Datasets

### 1.1 - name_basics

In [2]:
file_path = r"C:\Users\arada\Desktop\אוניברסיטה\הנדסה תעשייה וניהול\שנה ג\סמסטר ב\כרייה וניתוח נתונים מתקדם בפייתון\פרוייקט\DATA\name.basics.tsv.gz"
name_basics= pd.read_csv(
    file_path,
    sep="\t",
    low_memory=False,
    na_values="\\N"
)

name_basics.head()

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899.0,1987.0,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,1924.0,2014.0,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0045891"
2,nm0000003,Brigitte Bardot,1934.0,2025.0,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949.0,1982.0,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918.0,2007.0,"writer,director,actor","tt0050986,tt0069467,tt0050976,tt0083922"


### 1.2 - title_basics

In [3]:
file_path_2 = r"C:\Users\arada\Desktop\אוניברסיטה\הנדסה תעשייה וניהול\שנה ג\סמסטר ב\כרייה וניתוח נתונים מתקדם בפייתון\פרוייקט\DATA\title.basics.tsv.gz"

title_basics = pd.read_csv(
    file_path_2,
    sep="\t",
    low_memory=False,
    na_values="\\N"
)

title_basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1,Short


### 1.3 - title_principals

#### Heavy dataset; loaded only 1,000 rows to inspect the data structure

In [4]:
file_path_3 = r"C:\Users\arada\Desktop\אוניברסיטה\הנדסה תעשייה וניהול\שנה ג\סמסטר ב\כרייה וניתוח נתונים מתקדם בפייתון\פרוייקט\DATA\title.principals.tsv.gz"

title_principals_sample = pd.read_csv(
    file_path_3,
    sep="\t",
    nrows=1000,
    na_values="\\N",
    low_memory=False
)


print(title_principals_sample.shape)
title_principals_sample.head()

(1000, 6)


,tconst,ordering,nconst,category,job,characters
0,tt0000001,1,nm1588970,self,NaN,"[""Self""]"
1,tt0000001,2,nm0005690,director,NaN,NaN
2,tt0000001,3,nm0005690,producer,producer,NaN
3,tt0000001,4,nm0374658,cinematographer,director of photography,NaN
4,tt0000002,1,nm0721526,director,NaN,NaN


#### Loaded heavy all the dataset in chunks, keeping only relevant columns to save memory

In [5]:
chunk_size = 500000 


chunks = pd.read_csv(
    file_path_3, 
    sep="\t", 
    usecols=["tconst", "nconst", "category"], 
    na_values="\\N", 
    low_memory=False, 
    chunksize=chunk_size
)


all_chunks_list = []

for chunk in chunks:

    all_chunks_list.append(chunk)


title_principals = pd.concat(all_chunks_list, axis=0)

print(title_principals.shape)

(99129460, 3)


In [6]:
title_principals.head()

,tconst,nconst,category
0,tt0000001,nm1588970,self
1,tt0000001,nm0005690,director
2,tt0000001,nm0005690,producer
3,tt0000001,nm0374658,cinematographer
4,tt0000002,nm0721526,director


### 1.4 - title_ratings

In [7]:
file_path_4= r"C:\Users\arada\Desktop\אוניברסיטה\הנדסה תעשייה וניהול\שנה ג\סמסטר ב\כרייה וניתוח נתונים מתקדם בפייתון\פרוייקט\DATA\title.ratings.tsv.gz"
title_ratings = pd.read_csv(
    file_path_4,
    sep="\t",
    low_memory=False,
    na_values="\\N"
)

title_ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2211
1,tt0000002,5.5,316
2,tt0000003,6.4,2322
3,tt0000004,5.1,199
4,tt0000005,6.2,3046


## Part 2 - Filtering the datasets according to the required filters

#### 2.1 - runtimeMinutes between 60 and 300 & title Type == 'movie' & startYear <= 2024

In [8]:
title_basics['runtimeMinutes'] = pd.to_numeric(title_basics['runtimeMinutes'], errors='coerce')

title_basics = title_basics[
    (title_basics['titleType'] == 'movie') & 
    (title_basics['runtimeMinutes'].between(60, 300))&
    ((title_basics['startYear']) <= 2024)
]



##### Tracking the results

In [9]:
print(f"Number of movies after filtering: {title_basics.shape[0]}")
print(title_basics.head())

Number of movies after filtering: 388839
        tconst titleType                    primaryTitle  \
144  tt0000147     movie   The Corbett-Fitzsimmons Fight   
498  tt0000502     movie                        Bohemios   
570  tt0000574     movie     The Story of the Kelly Gang   
587  tt0000591     movie                The Prodigal Son   
672  tt0000679     movie  The Fairylogue and Radio-Plays   

                      originalTitle  isAdult  startYear  endYear  \
144   The Corbett-Fitzsimmons Fight        0     1897.0      NaN   
498                        Bohemios        0     1905.0      NaN   
570     The Story of the Kelly Gang        0     1906.0      NaN   
587               L'enfant prodigue        0     1907.0      NaN   
672  The Fairylogue and Radio-Plays        0     1908.0      NaN   

     runtimeMinutes                      genres  
144           100.0      Documentary,News,Sport  
498           100.0                         NaN  
570            70.0  Action,Adventure,B

##### Verifying that the conditions have been applied

In [10]:
print("--- All unique types in titleType ---")
print(title_basics['titleType'].unique())

print("\n--- Minimum Runtime ---")
print(f"Minimum: {title_basics['runtimeMinutes'].min()} minutes")

print("\n--- Maximum Runtime ---")
print(f"Maximum: {title_basics['runtimeMinutes'].max()} minutes")

print("\n--- Maximum Year ---")
print(f"Maximum Year: {title_basics['startYear'].max()}")

--- All unique types in titleType ---
['movie']

--- Minimum Runtime ---
Minimum: 60.0 minutes

--- Maximum Runtime ---
Maximum: 300.0 minutes

--- Maximum Year ---
Maximum Year: 2024.0


#### 2.2 - filtering by primaryTitle : Am -Az

Applying a regular expression

In [11]:
# The regex pattern
pattern = r'^A[m-zM-Z]'
title_basics = title_basics[title_basics['primaryTitle'].str.contains(pattern, na=False, regex=True)]

##### Tracking the results

In [12]:
print(f"Number of filtered movies: {title_basics.shape[0]}")
print(title_basics[['tconst', 'primaryTitle']].head(10))

Number of filtered movies: 11693
         tconst                  primaryTitle
2002  tt0002026  Anny - Story of a Prostitute
2598  tt0002625                    Ana Kadova
2619  tt0002646                      Atlantis
3596  tt0003632                      Arme Eva
3601  tt0003637                 Assunta Spina
6278  tt0006356             America Preparing
6295  tt0006373                  Arsene Lupin
6548  tt0006631          An Enemy to the King
7530  tt0007636              American Methods
8254  tt0008374                    Az obsitos


## Part 3 - Merging the datasets and selecting the relevant columns


<div class="alert alert-block alert-info">
<b>💡 Note:</b> At this stage, we will only merge the datasets. The Wikipedia fields will be appended later on.
</div>

#### 3.1 - Inner Join of Title Basics and Principals

In [13]:
basics_with_principals = pd.merge(
    title_basics, 
    title_principals, 
    on='tconst', 
    how='inner'
)

##### Tracking the results

In [14]:
print(f"Total rows after joining basics and principals: {basics_with_principals.shape[0]}")
print(basics_with_principals.head())

unique_movies_count = basics_with_principals['tconst'].nunique()
print(f"Number of movies: {unique_movies_count}")

Total rows after joining basics and principals: 169423
      tconst titleType                  primaryTitle  \
0  tt0002026     movie  Anny - Story of a Prostitute   
1  tt0002026     movie  Anny - Story of a Prostitute   
2  tt0002026     movie  Anny - Story of a Prostitute   
3  tt0002026     movie  Anny - Story of a Prostitute   
4  tt0002026     movie  Anny - Story of a Prostitute   

               originalTitle  isAdult  startYear  endYear  runtimeMinutes  \
0  Anny - en gatepiges roman        0     1912.0      NaN            68.0   
1  Anny - en gatepiges roman        0     1912.0      NaN            68.0   
2  Anny - en gatepiges roman        0     1912.0      NaN            68.0   
3  Anny - en gatepiges roman        0     1912.0      NaN            68.0   
4  Anny - en gatepiges roman        0     1912.0      NaN            68.0   

          genres     nconst category  
0  Drama,Romance  nm0418086  actress  
1  Drama,Romance  nm0027708    actor  
2  Drama,Romance  nm0526167 

#### 3.2 - Adding name_basics to the Merged Data

In [15]:
basics_principals_names = pd.merge(
    basics_with_principals, 
    name_basics, 
    on='nconst', 
    how='inner'
)

##### Tracking the results

In [16]:
print(f"Total rows after joining names: {basics_principals_names.shape[0]}")
unique_movies_count = basics_principals_names['tconst'].nunique()
print(f"Number of unique movies: {unique_movies_count}")

Total rows after joining names: 169421
Number of unique movies: 11561


#### 3.3 - Adding title_ratings to the Merged Data

<div class="alert alert-block" style="background-color: #f3e5f5; color: #4a148c; border-left: 6px solid #8e24aa;">
<b>📋 Note:</b> We used a left join to ensure we don't lose any data, as some movies in our dataset may not have ratings.
</div>

In [17]:
full_merged_df = pd.merge(
    basics_principals_names, 
    title_ratings, 
    on='tconst', 
    how='left'
)

##### Tracking the results

In [18]:
print(f"Total rows after joining ratings: {full_merged_df.shape[0]}")
unique_movies_count = full_merged_df['tconst'].nunique()
print(f"Number of unique movies: {unique_movies_count}")

Total rows after joining ratings: 169421
Number of unique movies: 11561


#### 3.4 Selecting Relevant Columns

In [19]:
movie_df = full_merged_df[[
    'tconst',
    'nconst',
    'primaryTitle',
    'startYear',
    'genres',
    'runtimeMinutes',
    'averageRating',
    'numVotes',
    'category',

]]

##### Tracking the results

In [20]:
print(f"Total rows in movie_df: {movie_df.shape[0]}")
print(f"Total columns in movie_df: {movie_df.shape[1]}")
unique_movies_count = movie_df['tconst'].nunique()
print(f"Number of unique movies: {unique_movies_count}")

Total rows in movie_df: 169421
Total columns in movie_df: 9
Number of unique movies: 11561


In [21]:
movie_df.head()

,tconst,nconst,primaryTitle,startYear,genres,runtimeMinutes,averageRating,numVotes,category
0,tt0002026,nm0418086,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actress
1,tt0002026,nm0027708,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor
2,tt0002026,nm0526167,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actress
3,tt0002026,nm0959066,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor
4,tt0002026,nm0064944,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor


## Part 4 - Creating a Column for the Top 5 Actors

<div class="alert alert-block" style="background-color: #fffde7; color: #b71c1c; border-left: 6px solid #f44336;">
<b> Note:</b> Although the project requirements specified extracting "actors" only, we have also included actresses.
</div>

In [22]:
movie_df = full_merged_df[[
    'tconst',
    'nconst',
    'primaryTitle',
    'startYear',
    'genres',
    'runtimeMinutes',
    'averageRating',
    'numVotes',
    'category'
]].copy()


actors_only = movie_df[movie_df['category'].isin(['actor', 'actress'])]


top_5_actors = actors_only.groupby('tconst')['nconst'].apply(lambda x: list(x)[:5])


movie_df['lead_actors_ids'] = movie_df['tconst'].map(top_5_actors)


movie_df['lead_actors_ids'] = movie_df['lead_actors_ids'].apply(lambda x: x if isinstance(x, list) else [])

##### Sanity Check

In [23]:
clean_list = movie_df[movie_df['tconst'] == 'tt0002026']['lead_actors_ids'].values[0]
print(clean_list)

['nm0418086', 'nm0027708', 'nm0526167', 'nm0959066', 'nm0064944']


## Part 5 - Removing Duplicates Keeping One Row per Movie

In [24]:
movie_df.head(10)

,tconst,nconst,primaryTitle,startYear,genres,runtimeMinutes,averageRating,numVotes,category,lead_actors_ids
0,tt0002026,nm0418086,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actress,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
1,tt0002026,nm0027708,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
2,tt0002026,nm0526167,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actress,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
3,tt0002026,nm0959066,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
4,tt0002026,nm0064944,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
5,tt0002026,nm0348052,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actress,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
6,tt0002026,nm0959065,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actress,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
7,tt0002026,nm0115982,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
8,tt0002026,nm0804396,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
9,tt0002026,nm0804396,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,actor,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."


<div class="alert alert-block" style="background-color: #fff3e0; color: #e65100; border-left: 6px solid #ff9800;">
<b>💡 Note:</b> As observed, multiple rows exist for a single movie. This duplication occurs because the <code>nconst</code> field introduces multiple entries per movie.
</div>

In [25]:
movie_df = movie_df[[
    'tconst',
    'primaryTitle',
    'startYear',
    'genres',
    'runtimeMinutes',
    'averageRating',
    'numVotes',
    'lead_actors_ids'
]]

##### Tracking the results

In [26]:
movie_df = movie_df.drop_duplicates(subset=['tconst']).reset_index(drop=True)
print(f"Total rows in movie_df (movies only): {movie_df.shape[0]}")
print(f"Total columns in movie_df: {movie_df.shape[1]}")

Total rows in movie_df (movies only): 11561
Total columns in movie_df: 8


In [27]:
movie_df.head()

,tconst,primaryTitle,startYear,genres,runtimeMinutes,averageRating,numVotes,lead_actors_ids
0,tt0002026,Anny - Story of a Prostitute,1912.0,"Drama,Romance",68.0,4.6,19.0,"[nm0418086, nm0027708, nm0526167, nm0959066, n..."
1,tt0216541,Andrine og Kjell,1952.0,"Drama,Romance",85.0,5.6,45.0,"[nm0392572, nm0026163, nm0561273, nm0835611, n..."
2,tt0002625,Ana Kadova,1913.0,NaN,92.0,NaN,NaN,"[nm0012481, nm0140054, nm0243918, nm0261907, n..."
3,tt0002646,Atlantis,1913.0,Drama,121.0,6.5,526.0,"[nm0299761, nm0650120, nm0860895, nm0491498, n..."
4,tt0129944,Art and the Woman,1922.0,Drama,105.0,NaN,NaN,"[nm0414716, nm0814379, nm0927378, nm0299757, n..."


<div class="alert alert-block" style="background-color: #e8f5e9; color: #1b5e20; border-left: 6px solid #4caf50; text-align: center;">
<b>✅ Note:</b> IMDB part is officially done!
</div>

# Part 6 - Web Scraping From Wikipedia

### Before scraping Wikipedia for the entire dataset, we will run a trial on the first 20 rows to test the script

In [28]:
HEADERS = {
    "User-Agent": "MovieDataScraper/1.0 (learning_project@example.com) - Python Script"
}

def money_to_millions(text):
    try:
        if pd.isna(text): 
            return np.nan
        text = str(text).replace(",", "").replace("\xa0", " ").lower()
        nums = re.findall(r'\d+(?:\.\d+)?', text)
        if not nums: 
            return np.nan
        nums = [float(x) for x in nums]
        value = sum(nums[:2]) / 2 if len(nums) >= 2 else nums[0]

        if "billion" in text: 
            value *= 1000
        elif "million" in text: 
            pass
        else: 
            value /= 1_000_000
        return value
    except:
        return np.nan

def get_movie_data(title, year):
    try:
        api_url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "list": "search",
            "srsearch": f"{title} {int(year)} film",
            "format": "json"
        }

        res = requests.get(api_url, params=params, headers=HEADERS, timeout=10)
        
        # Verify that Wikipedia returned a successful response (status code 200) before parsing JSON
        if res.status_code != 200:
            print(f" Wikipedia server error (status code {res.status_code}).")
            return pd.Series([np.nan]*5)

        search_data = res.json()

        # If no search results are found
        if not search_data.get("query", {}).get("search"):
            print(" Wikipedia page not found.")
            return pd.Series([np.nan]*5)

        # Retrieve the title of the first search result
        page_title = search_data["query"]["search"][0]["title"]
        print(f" Found page: {page_title}")

        # Fetch the actual page HTML to find the data table
        wiki_url = f"https://en.wikipedia.org/wiki/{page_title.replace(' ', '_')}"
        res = requests.get(wiki_url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        language, country, budget, boxoffice, plot = np.nan, np.nan, np.nan, np.nan, np.nan

        # info-box
        infobox = soup.find("table", class_=re.compile("infobox"))

        if infobox:
            for row in infobox.find_all("tr"):
                header = row.find("th")
                value = row.find("td")

                if header and value:
                    key = header.get_text(" ", strip=True).lower()
                    val = value.get_text(" ", strip=True)

                    if "language" in key: 
                        language = val
                    elif "country" in key: 
                        country = val
                    elif "budget" in key: 
                        budget = money_to_millions(val)
                    elif "box office" in key: 
                        boxoffice = money_to_millions(val)
        else:
            print(" No data table (Infobox) found on this page.")

        # movie plot 
        for p in soup.find_all("p"):
            text = p.get_text(" ", strip=True)
            if len(text) > 100:
                plot = text
                break

        return pd.Series([language, country, budget, boxoffice, plot])

    except Exception as e:
        print(f" Error retrieving data: {e}")
        return pd.Series([np.nan] * 5)

In [29]:
# Using 20 random movies from movie_df
test_df = movie_df.sample(n=20, random_state=42).copy()

for col in ["Language", "Country", "budget", "BoxOffice", "plot"]:
    test_df[col] = np.nan

total_movies = len(test_df)
print(f" Starting scan of {total_movies}\n")

# Use a running counter for the backup mechanism
counter = 0

# Iterate row by row to track exact progress
for idx, row in test_df.iterrows():
    title = row["primaryTitle"]
    year = row["startYear"]
    counter += 1
    
    # Clean output: Just saying we are loading the specific movie
    print(f"Retrieving data for: {title} ({year})...")
    
    data = get_movie_data(title, year)
    test_df.loc[idx, ["Language", "Country", "budget", "BoxOffice", "plot"]] = data.values
    
    # Backup mechanism: saves progress every 5 movies
    if counter % 5 == 0:
        test_df.to_csv('movie_df_backup.csv', index=False, encoding='utf-8')
        print(f" Interim backup saved successfully (processed {counter} movies)\n")
    
    # 1-second delay between requests to avoid getting blocked
    time.sleep(1)

# Final save of the entire file after the loop finishes
test_df.to_csv('movie_df_final.csv', index=False, encoding='utf-8')
print("\n Scan completed successfully! All data has been saved to: movie_df_final.csv")


 Starting scan of 20

Retrieving data for: America Becoming (1991.0)...
 Found page: Captain America (1990 film)
Retrieving data for: Andromina: The Pleasure Planet (2000.0)...
 Wikipedia page not found.
Retrieving data for: Are You Ready? (2017.0)...
 Found page: Ready Player One (film)
Retrieving data for: Ar meno un quejío (2005.0)...
 Found page: Vicenta Ndongo
Retrieving data for: Az alvilág professzora (1969.0)...
 Found page: Zoltán Latinovits
 Interim backup saved successfully (processed 5 movies)

Retrieving data for: Attraction (2016.0)...
 Found page: The Rules of Attraction (film)
Retrieving data for: Armastuse lahinguväljad (1992.0)...
 Found page: List of Estonian films since 1991
 No data table (Infobox) found on this page.
Retrieving data for: Anba hachi tengu (1961.0)...
 Wikipedia page not found.
Retrieving data for: Amin (2016.0)...
 Found page: The Last King of Scotland (film)
Retrieving data for: As If They Were Angels (2018.0)...
 Found page: Hell's Angels (film)


In [30]:
test_df[["primaryTitle", "startYear", "Language", "Country", "budget", "BoxOffice", "plot"]]

,primaryTitle,startYear,Language,Country,budget,BoxOffice,plot
5629,America Becoming,1991.0,English,United States,2.5,NaN,Captain America is a 1990 American superhero f...
3710,Andromina: The Pleasure Planet,2000.0,NaN,NaN,NaN,NaN,NaN
11254,Are You Ready?,2017.0,English,United States [ 5 ],165.0,308.45,Ready Player One is a 2018 American science fi...
5636,Ar meno un quejío,2005.0,NaN,NaN,NaN,NaN,Vicenta Ndongo (born 23 February 1968) is a Sp...
1491,Az alvilág professzora,1969.0,NaN,NaN,NaN,NaN,"Zoltán Latinovits (9 September 1931, in Budape..."
7958,Attraction,2016.0,English,NaN,3.0,6.90,The Rules of Attraction is a 2002 dark comedy ...
6594,Armastuse lahinguväljad,1992.0,NaN,NaN,NaN,NaN,A list of films made in the Republic of Estoni...
4802,Anba hachi tengu,1961.0,NaN,NaN,NaN,NaN,NaN
11365,Amin,2016.0,English Swahili,NaN,5.0,26.20,The Last King of Scotland is a 2006 historical...
6951,As If They Were Angels,2018.0,English,United States,2.4,2.75,Hell's Angels is a 1930 American pre-Code inde...


### Wikipedia Web Scraping on the Full Dataset

<div class="alert alert-block" style="background-color: #ffebee; color: #c62828; border-left: 6px solid #f44336; padding: 15px; border-radius: 4px; text-align: center;">
<b>⚠️ WARNING:</b><br>
Running this script on the entire dataset can take a very long time. It is highly recommended to skip this step and proceed to the next section, where we load the dataset generated after the run.
</div>

In [ ]:
# Copy the entire movie_df dataset
full_df = movie_df.copy() 

# Initialize new columns with NaN
for col in ["Language", "Country", "budget", "BoxOffice", "plot"]:
    full_df[col] = np.nan

total_movies = len(full_df)
print(f"Starting scan of {total_movies} movies. This will take some time")

# Iterate row by row to track exact progress and handle potential hangs
for idx, row in full_df.iterrows():
    title = row["primaryTitle"]
    year = row["startYear"]
    
    print(f"\nScanning [{idx+1}/{total_movies}]: {title} ({year})...")
    
    data = get_movie_data(title, year)
    full_df.loc[idx, ["Language", "Country", "budget", "BoxOffice", "plot"]] = data.values
    
    # Backup mechanism: saves progress every 100 movies
    if (idx + 1) % 100 == 0:
        full_df.to_csv('movie_df_backup.csv', index=False, encoding='utf-8')
        print(f"Interim backup saved successfully (up to movie {idx+1})")
    
    # 1-second delay between requests to avoid getting blocked (scraping best practice)
    time.sleep(1)

# Final save of the entire dataset after the loop completes
full_df.to_csv('movie_df_final.csv', index=False, encoding='utf-8')
print("\nScan completed successfully! All data has been saved to: movie_df_final.csv")

<div class="alert alert-block" style="background-color: #e8f5e9; color: #1b5e20; border-left: 6px solid #4caf50; text-align: center;">
<b></b> Wikipedia Web Scraping part has been completed successfully ✅
</div>

# Part 7 - Loading the Dataset After Web Scraping & Cleaning the Dirty Data

### 7.1 - Loading the Dataset

In [31]:
movie_df_file_path = r"C:\Users\arada\Desktop\אוניברסיטה\הנדסה תעשייה וניהול\שנה ג\סמסטר ב\כרייה וניתוח נתונים מתקדם בפייתון\פרוייקט\movie_df_final.csv"
movie_df = pd.read_csv(movie_df_file_path)

### 7.2 - Cleaning the Dirty Data

####  7.2.1 - Cleaning the Language Column

##### Inspecting the Unique Values

In [32]:
if 'Language' in movie_df.columns:
    unique_languages = movie_df['Language'].unique()
    print(f" Column 'Language' ({len(unique_languages)} unique values):")
    print(unique_languages)
else:
    print(" Column 'Language' not found in movie_df ")

 Column 'Language' (454 unique values):
['Norwegian' nan 'silent film Danish intertitles'
 'Silent film Italian intertitles' 'Silent English intertitles'
 'Silent (English intertitles )' 'English [ 4 ]'
 'Silent (English subtitles)' 'English' 'Silent film' 'French' 'Silent'
 'Silent Swedish intertitles' 'Silent ( English intertitles)'
 'Silent..English titles' 'Silent German intertitles'
 'Silent French intertitles' 'Sound (Synchronized) English Intertitles'
 'Silent with English intertitles'
 'Silent film Russian and Ukrainian intertitles'
 'Sound (All-Talking) English' 'Russian' 'German' 'Italian' 'Swedish'
 'Japanese' 'Hindi and Marathi' 'Hungarian'
 'Silent film Japanese intertitles' 'Japanese German' 'Spanish'
 'Portuguese' 'Hindustani [ 1 ]' 'Hindustani' 'English Greek'
 'English Italian' 'English German' 'Hindi' 'Italian English' 'Hindi Urdu'
 'Tamil' 'English Japanese' 'Bengali' 'English French' 'Malay' 'Polish'
 'German English' 'Serbo-Croatian' 'Georgian' 'Greek' 'Telugu' 'Du

<div class="alert alert-block" style="background-color: #fffde7; color: #4a148c; border-left: 6px solid #8e24aa; padding: 20px; border-radius: 6px; text-align: center;">
    <h3 style="margin-top: 0; color: #4a148c; font-weight: bold;">
        💡 Cleaning Logic for 'Language' Column
    </h3>
    <p style="margin: 10px 0 0 0; font-size: 16px; line-height: 1.5;">
        We keep only alphanumeric characters, spaces, percentages, and hyphens from the start of the string, <br>
        and discard everything to the right of the first encountered invalid character (such as brackets, parentheses, or commas).
    </p>
</div>
</div>

In [33]:
def clean_string(text):
    # If the value is NaN or not a string, return as is
    if pd.isna(text) or not isinstance(text, str):
        return text
    
    # Match only allowed characters from the start of the string:
    # a-z, A-Z, 0-9, % (percent), \s (whitespace), and - (hyphen)
    match = re.match(r"^([a-zA-Z0-9%\s\-]*)", text)
    
    if match:
        # Extract the valid part and clean trailing/leading spaces
        return match.group(1).strip()
    
    return ""

# Apply the cleaning function to the 'Language' column in movie_df
if 'Language' in movie_df.columns:
    movie_df['Language'] = movie_df['Language'].apply(clean_string)
    print(" 'Language' column cleaned successfully (including hyphens)!")
else:
    print(" Column 'Language' not found in movie_df ")

 'Language' column cleaned successfully (including hyphens)!


##### Tracking the results

In [34]:
if 'Language' in movie_df.columns:
    unique_languages = movie_df['Language'].unique()
    print(f" Column 'Language' ({len(unique_languages)} unique values):")
    print(unique_languages)
else:
    print(" Column 'Language' not found in movie_df ")

 Column 'Language' (360 unique values):
['Norwegian' nan 'silent film Danish intertitles'
 'Silent film Italian intertitles' 'Silent English intertitles' 'Silent'
 'English' 'Silent film' 'French' 'Silent Swedish intertitles'
 'Silent German intertitles' 'Silent French intertitles' 'Sound'
 'Silent with English intertitles'
 'Silent film Russian and Ukrainian intertitles' 'Russian' 'German'
 'Italian' 'Swedish' 'Japanese' 'Hindi and Marathi' 'Hungarian'
 'Silent film Japanese intertitles' 'Japanese German' 'Spanish'
 'Portuguese' 'Hindustani' 'English Greek' 'English Italian'
 'English German' 'Hindi' 'Italian English' 'Hindi Urdu' 'Tamil'
 'English Japanese' 'Bengali' 'English French' 'Malay' 'Polish'
 'German English' 'Serbo-Croatian' 'Georgian' 'Greek' 'Telugu' 'Dutch'
 'none' 'Slovene' 'Russian Tatar Italian' '90 minutes'
 'French German English' 'See regional official languages' 'Finnish'
 'Cantonese' 'English French Spanish' 'Croatian' 'Dakhani'
 'Italian Arabic' 'French English'

#### 7.2.2 - Cleaning the Country Column

##### Inspecting the Unique Values

In [35]:
if 'Country' in movie_df.columns:
    unique_countries = movie_df['Country'].unique()
    print(f" Column 'Country' ({len(unique_countries)} unique values):")
    print(unique_countries)
else:
    print(" Column 'Country' not found in movie_df ")

 Column 'Country' (162 unique values):
['Norway' nan 'Denmark' 'Italy' 'United Kingdom' 'United States'
 'United States [ 3 ]' 'Germany' 'Sweden' 'USA' 'France'
 'United States of America' 'Soviet Union' 'Germany [ 1 ]' 'Japan' 'India'
 'Nazi Germany' 'Hungary' 'West Germany' 'Spain' 'Brazil' 'Portugal'
 'Mexico' 'British India' 'Czechoslovakia' 'Argentina' 'Austria'
 'United States [ 2 ]' 'Australia' 'East Germany'
 'United Kingdom United States [ 1 ]' 'Venezuela' 'Poland' 'Netherlands'
 'FPR Yugoslavia' 'Italy [ 1 ]' 'Italy [ 2 ]' 'West Germany [ 1 ]'
 'Greece' 'Canada' 'Yugoslavia' 'United Kingdom / France' 'Finland'
 'Italy [ 3 ]' 'SFR Yugoslavia' 'Bulgaria' 'Hong Kong' 'Israel'
 'Australia Taiwan [ 1 ]' 'Egypt' 'New Zealand' 'United States [ 1 ]'
 'United States [ 1 ] [ 2 ]' 'Sri Lanka' 'France [ 1 ]' 'Belgium'
 'North Macedonia' 'Faroe Islands' 'Chile' 'Estonia' 'Philippines'
 'Russia' 'Iceland' 'Iran' 'Ivory Coast' 'Slovakia' 'British Hong Kong'
 'Singapore' 'Canada [ 3 ]' 'Czec

<div class="alert alert-block" style="background-color: #e3f2fd; color: #0d47a1; border-left: 6px solid #2196f3; padding: 20px; border-radius: 6px; text-align: center;">
    <h3 style="margin-top: 0; color: #0d47a1; font-weight: bold; text-align: center;">
        💡 Cleaning Logic for 'Country' Column
    </h3>
    <p style="margin: 10px 0 0 0; font-size: 16px; line-height: 1.5; text-align: center;">
        Same cleaning logic as applied to the 'Language' column.
    </p>
</div>
</div>
</div>

In [36]:
# Apply the cleaning function to the 'Country' column in movie_df
if 'Country' in movie_df.columns:
    movie_df['Country'] = movie_df['Country'].apply(clean_string)
    print(" 'Country' column cleaned successfully (including hyphens)!")
else:
    print(" Column 'Country' not found in movie_df.")

 'Country' column cleaned successfully (including hyphens)!


##### Tracking the results

In [37]:
if 'Country' in movie_df.columns:
    unique_countries = movie_df['Country'].unique()
    print(f" Column 'Country' ({len(unique_countries)} unique values):")
    print(unique_countries)
else:
    print(" Column 'Country' not found in movie_df.")

 Column 'Country' (119 unique values):
['Norway' nan 'Denmark' 'Italy' 'United Kingdom' 'United States' 'Germany'
 'Sweden' 'USA' 'France' 'United States of America' 'Soviet Union' 'Japan'
 'India' 'Nazi Germany' 'Hungary' 'West Germany' 'Spain' 'Brazil'
 'Portugal' 'Mexico' 'British India' 'Czechoslovakia' 'Argentina'
 'Austria' 'Australia' 'East Germany' 'United Kingdom United States'
 'Venezuela' 'Poland' 'Netherlands' 'FPR Yugoslavia' 'Greece' 'Canada'
 'Yugoslavia' 'Finland' 'SFR Yugoslavia' 'Bulgaria' 'Hong Kong' 'Israel'
 'Australia Taiwan' 'Egypt' 'New Zealand' 'Sri Lanka' 'Belgium'
 'North Macedonia' 'Faroe Islands' 'Chile' 'Estonia' 'Philippines'
 'Russia' 'Iceland' 'Iran' 'Ivory Coast' 'Slovakia' 'British Hong Kong'
 'Singapore' 'Czech Republic' 'China' 'Northern Ireland' 'U' 'North Korea'
 'Turkey' 'Ireland' 'England' 'Pakistan' 'South Korea' 'Peru' 'Tajikistan'
 'Korea' 'Malaysia' 'Algeria' 'Indonesia' 'Thailand' 'DR Congo' 'Niger'
 'Colombia' 'South Africa' 'Taiwan' 'Lith

####  7.2.3- Cleaning the budget Column

<div class="alert alert-block" style="background-color: #f3e5f5; color: #4a148c; border-left: 6px solid #ab47bc; padding: 20px; border-radius: 6px; text-align: center;">
    <h3 style="margin-top: 0; color: #4a148c; font-weight: bold; text-align: center;">
        💡 Validation Logic for 'budget' Column
    </h3>
    <p style="margin: 10px 0 0 0; font-size: 16px; line-height: 1.5; text-align: center;">
        We filter out missing values (NaN) and attempt to cast each remaining value in the 'budget' column to a float; <br>
        if the conversion fails, we flag and count it as a "dirty" or non-numeric value.
    </p>
</div>

In [38]:
def floot_Validation(df, column_name):
    if column_name in df.columns:
        non_null_values = df[column_name].dropna()
        
        invalid_count = 0
        invalid_values = []

        for val in non_null_values:
            try:
                float(val)
            except (ValueError, TypeError):
                invalid_count += 1
                invalid_values.append(val)

        print(f"Column checked: {column_name}")
        print(f"Total values that cannot be converted to float: {invalid_count}")
        
        if invalid_count > 0:
            print(f"Sample of problematic values (up to 10): {invalid_values[:10]}")
    else:
        print(f"Column '{column_name}' not found in DataFrame.")

In [39]:
floot_Validation(movie_df, "budget")

Column checked: budget
Total values that cannot be converted to float: 0


####  7.2.3- Cleaning the BoxOffice Column

<div class="alert alert-block" style="background-color: #fff3e0; color: #e65100; border-left: 6px solid #ff9800; padding: 20px; border-radius: 6px; text-align: center;">
    <h3 style="margin-top: 0; color: #e65100; font-weight: bold; text-align: center;">
        💡 Validation Logic for 'BoxOffice' Column
    </h3>
    <p style="margin: 10px 0 0 0; font-size: 16px; line-height: 1.5; text-align: center;">
        Same validation logic as applied to the 'budget' column.
    </p>
</div>

In [40]:
floot_Validation(movie_df, "BoxOffice")

Column checked: BoxOffice
Total values that cannot be converted to float: 0


<div class="alert alert-block" style="background-color: #e8f5e9; color: #1b5e20; border-left: 6px solid #4caf50; padding: 20px; border-radius: 6px; text-align: center;">
    <h3 style="margin-top: 0; color: #1b5e20; font-weight: bold; text-align: center;">
         Data Cleaning Completed Successfully ✅
    </h3>
    <p style="margin: 10px 0 0 0; font-size: 16px; line-height: 1.5; text-align: center;">
        All target columns have been processed, cleaned, and validated.
    </p>
</div>

# Part 8 - Changing Column Data Types

### Checking Column Data Types

In [41]:
movie_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11561 entries, 0 to 11560
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tconst           11561 non-null  object 
 1   primaryTitle     11561 non-null  object 
 2   startYear        11561 non-null  int64  
 3   genres           10888 non-null  object 
 4   runtimeMinutes   11561 non-null  int64  
 5   averageRating    8308 non-null   float64
 6   numVotes         8308 non-null   float64
 7   lead_actors_ids  11561 non-null  object 
 8   Language         5367 non-null   object 
 9   Country          4536 non-null   object 
 10  budget           2051 non-null   float64
 11  BoxOffice        2348 non-null   float64
 12  plot             10178 non-null  object 
dtypes: float64(4), int64(2), object(7)
memory usage: 1.1+ MB


### Converting Columns to Their Appropriate Data Types

In [42]:
# Convert object columns to string
if 'tconst' in movie_df.columns:
    movie_df['tconst'] = movie_df['tconst'].astype('string')
    print(" 'tconst' successfully converted to string.")
else:
    print(" Column 'tconst' not found.")

if 'primaryTitle' in movie_df.columns:
    movie_df['primaryTitle'] = movie_df['primaryTitle'].astype('string')
    print(" 'primaryTitle' successfully converted to string.")
else:
    print(" Column 'primaryTitle' not found.")

if 'Language' in movie_df.columns:
    movie_df['Language'] = movie_df['Language'].astype('string')
    print(" 'Language' successfully converted to string.")
else:
    print(" Column 'Language' not found.")

if 'Country' in movie_df.columns:
    movie_df['Country'] = movie_df['Country'].astype('string')
    print(" 'Country' successfully converted to string.")
else:
    print(" Column 'Country' not found.")

if 'plot' in movie_df.columns:
    movie_df['plot'] = movie_df['plot'].astype('string')
    print(" 'plot' successfully converted to string.")
else:
    print(" Column 'plot' not found.")


# Convert numVotes from float to int
if 'numVotes' in movie_df.columns:
    movie_df['numVotes'] = pd.to_numeric(
        movie_df['numVotes'],
        errors='coerce'
    ).fillna(0).astype('int64')

    print(" 'numVotes' successfully converted to int.")
else:
    print(" Column 'numVotes' not found.")

 'tconst' successfully converted to string.
 'primaryTitle' successfully converted to string.
 'Language' successfully converted to string.
 'Country' successfully converted to string.
 'plot' successfully converted to string.
 'numVotes' successfully converted to int.


##### Tracking the results

In [43]:
movie_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11561 entries, 0 to 11560
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tconst           11561 non-null  string 
 1   primaryTitle     11561 non-null  string 
 2   startYear        11561 non-null  int64  
 3   genres           10888 non-null  object 
 4   runtimeMinutes   11561 non-null  int64  
 5   averageRating    8308 non-null   float64
 6   numVotes         11561 non-null  int64  
 7   lead_actors_ids  11561 non-null  object 
 8   Language         5367 non-null   string 
 9   Country          4536 non-null   string 
 10  budget           2051 non-null   float64
 11  BoxOffice        2348 non-null   float64
 12  plot             10178 non-null  string 
dtypes: float64(3), int64(3), object(2), string(5)
memory usage: 1.1+ MB


<span style="color:red">
*In Pandas, a column containing a list of strings is represented as an object data type.
</span>

# Part 9 - Reviewing Missing Values

In [44]:
num_movies = movie_df['tconst'].count()

summary = pd.DataFrame({
    'missing_values': movie_df.isnull().sum(),
    'missing_values_percent': (movie_df.isnull().sum() / len(movie_df) * 100).round(2)
})

print(f"Number of movies: {num_movies}")
print(summary.sort_values(by='missing_values', ascending=False))

Number of movies: 11561
                 missing_values  missing_values_percent
budget                     9510                   82.26
BoxOffice                  9213                   79.69
Country                    7025                   60.76
Language                   6194                   53.58
averageRating              3253                   28.14
plot                       1383                   11.96
genres                      673                    5.82
tconst                        0                    0.00
primaryTitle                  0                    0.00
startYear                     0                    0.00
runtimeMinutes                0                    0.00
numVotes                      0                    0.00
lead_actors_ids               0                    0.00


# Part 10 - Create a CSV file of cleaned data


In [47]:
movie_df.to_csv('movie_df_cleaned_final.csv', index=False, encoding='utf-8')

print("file saved successfully on name: movie_df_cleaned_final")

file saved successfully on name: movie_df_cleaned_final


<div class="alert alert-block" style="background-color: #e8f5e9; color: #1b5e20; border-left: 6px solid #4caf50; padding: 20px; border-radius: 6px; text-align: center;">
    <h3 style="margin: 0; color: #1b5e20; font-weight: bold; text-align: center;">
        ✅ Part 1 Completed Successfully ✅
    </h3>
</div>